# Olist E-Commerce Dataset — Data Profiling

## Introduction

This notebook performs systematic **data profiling and quality assessment** of the Olist e-commerce marketplace dataset. The objective is to understand the structure, completeness, consistency, uniqueness, and relationships across the different tables before using the data for downstream analysis and machine learning.

The profiling covers the following tables:

- `orders`
- `customers`
- `order_items`
- `products`
- `sellers`
- `payments`
- `order_reviews`
- `geolocation`
- `category_translation`

## Profiling Approach

For each table, the following aspects were examined where applicable:

1. **Dataset structure** — number of rows, columns, column names, and data types.
2. **Missing values** — identification and assessment of null values.
3. **Uniqueness and duplicates** — identification of candidate keys, duplicate records, and repeated entities.
4. **Descriptive statistics** — distribution, ranges, and summary statistics of numerical variables.
5. **Categorical analysis** — unique values and frequency distributions of categorical variables.
6. **Data-quality checks** — identification of invalid, inconsistent, or suspicious values.
7. **Temporal checks** — examination of date ranges and consistency of timestamps where applicable.
8. **Logical consistency checks** — validation of relationships and business rules within individual tables.
9. **Cross-table relationship checks** — verification of key relationships, coverage, and unmatched records between related tables.
10. **Important profiling observations** — key findings and potential data-quality issues were recorded for each table.

## Outcome

The profiling provides a structured understanding of the dataset and identifies important characteristics such as missing values, duplicate records, key candidates, inconsistent values, outliers, relationship issues, and coverage gaps.

These findings will serve as the foundation for the **data preprocessing, feature engineering, exploratory data analysis, visualization, and machine learning stages** of the project.

In [1]:
#importing the required libraries
import pandas as pd
import numpy as np

In [9]:
#loading all the tables
customers = pd.read_csv("C:\\Users\\ravic\\Downloads\\olist_customers_dataset.csv")
orders = pd.read_csv("C:\\Users\\ravic\\Downloads\\olist_orders_dataset.csv")
order_items = pd.read_csv("C:\\Users\\ravic\\Downloads\\olist_order_items_dataset.csv")
order_payments = pd.read_csv("C:\\Users\\ravic\\Downloads\\olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("C:\\Users\\ravic\\Downloads\\olist_order_reviews_dataset.csv")
products = pd.read_csv("C:\\Users\\ravic\\Downloads\\olist_products_dataset.csv")
sellers = pd.read_csv("C:\\Users\\ravic\\Downloads\\olist_sellers_dataset.csv")
geolocation = pd.read_csv("C:\\Users\\ravic\\Downloads\\olist_geolocation_dataset.csv")
category_translation = pd.read_csv("C:\\Users\\ravic\\Downloads\\product_category_name_translation.csv")

In [112]:
#shape of each table
print(f"customers_shape : {customers.shape}")
print(f"orders_shape : {orders.shape}")
print(f"order_items_shape : {order_items.shape}")
print(f"order_payments_shape : {order_payments.shape}")
print(f"order_reviews_shape : {order_reviews.shape}")
print(f"products_shape : {products.shape}")
print(f"sellers_shape : {sellers.shape}")
print(f"geolocation_shape : {geolocation.shape}")
print(f"category_translation_shape : {category_translation.shape}")

customers_shape : (99441, 5)
orders_shape : (99441, 8)
order_items_shape : (112650, 7)
order_payments_shape : (103886, 5)
order_reviews_shape : (99224, 7)
products_shape : (32951, 9)
sellers_shape : (3095, 4)
geolocation_shape : (1000163, 5)
category_translation_shape : (71, 2)


In [46]:
#dataset summary for all tables
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

summary = []

for name, df in tables.items():
    summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

,table,rows,columns,missing_values,duplicate_rows
0,customers,99441,5,0,0
1,orders,99441,8,4908,0
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
4,order_reviews,99224,7,145903,0
5,products,32951,9,2448,0
6,sellers,3095,4,0,0
7,geolocation,1000163,5,0,261831
8,category_translation,71,2,0,0


### **Data Profiling Orders Table :**

In [28]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [20]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [22]:
orders.tail()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00
99440,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08 20:57:30,2018-03-09 11:20:28,2018-03-09 22:11:59,2018-03-16 13:08:30,2018-04-03 00:00:00


In [24]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

In [30]:
orders.describe(include="all")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-04-11 10:48:14,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 23:38:46,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [32]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [48]:
#order status where order_delivered_customer_date is null
orders.groupby("order_status")["order_delivered_customer_date"].apply(
    lambda x: x.isna().sum()
)

order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_delivered_customer_date, dtype: int64

In [42]:
#finding out those eight orders
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [60]:
#no of duplicated rows
orders.duplicated().sum()

0

In [66]:
#no of missing values
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [68]:
#percentage of missing values
orders.isnull().mean() * 100

order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
dtype: float64

In [70]:
orders["order_status"].unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

In [88]:
#to check logical consistency of dates
purchase = pd.to_datetime(orders["order_purchase_timestamp"])
approved = pd.to_datetime(orders["order_approved_at"])
carrier = pd.to_datetime(orders["order_delivered_carrier_date"])
customer_delivery = pd.to_datetime(orders["order_delivered_customer_date"])
estimated = pd.to_datetime(orders["order_estimated_delivery_date"])

In [74]:
(purchase > approved).sum()

0

In [80]:
(approved > carrier).sum()

1359

In [82]:
orders[
    (approved > carrier)
][[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].head(10)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04 00:00:00
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15 00:00:00
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31 00:00:00
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31 00:00:00
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06 00:00:00
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18,2018-07-23 00:00:00
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57,2018-08-06 00:00:00
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18,2018-06-07 00:00:00
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04 00:00:00
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28,2018-07-19 00:00:00


In [86]:
(carrier > customer_delivery).sum()

23

In [90]:
late = customer_delivery > estimated
late.sum()

7827

In [96]:
#separating the orders into on-time/early, late, and missing delivery date.
delivery_diff = customer_delivery - estimated
print("Early/on-time:", (delivery_diff <= pd.Timedelta(0)).sum())
print("Late:", (delivery_diff > pd.Timedelta(0)).sum())
print("Missing:", delivery_diff.isna().sum())

Early/on-time: 88649
Late: 7827
Missing: 2965


In [100]:
#date range
print("Purchase date range:")
print(purchase.min(), "to", purchase.max())

print("\nEstimated delivery date range:")
print(estimated.min(), "to", estimated.max())

Purchase date range:
2016-09-04 21:15:19 to 2018-10-17 17:30:18

Estimated delivery date range:
2016-09-30 00:00:00 to 2018-11-12 00:00:00


### **Profiling Findings — orders :** 

* **The table contains 99,441 orders and 8 columns.**
* **order_id is unique across all records and is a strong primary-key candidate.**
* **customer_id is also unique within the orders table.**
* **There are no completely duplicated rows.**
* **order_status contains 8 categories, with approximately 97% of orders marked as delivered.**
* **Missing values occur only in order lifecycle timestamps, with the highest missingness in order_delivered_customer_date (2.98%).**
* **Missing delivery dates are strongly associated with order status and therefore should not be blindly imputed or removed.**
* **8 delivered orders have missing customer delivery dates and represent potential data-quality anomalies.**
* **The five temporal columns are stored as object and will require datetime conversion during later preprocessing.**
* **Purchase-to-approval timestamps are temporally consistent.**
* **1,359 orders (approx.1.37%) show carrier timestamps earlier than approval timestamps, while 23 orders (approx.0.02%) show customer delivery earlier than carrier delivery. These are potential temporal anomalies requiring consideration later.**
* **7,827 orders (7.87%) were delivered after the estimated delivery date; among orders with a recorded delivery date, approximately 8.12% were late.**
* **Purchase timestamps span approximately September 2016 to October 2018.**



### **Data Profiling Customers Table :**

In [11]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [13]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [15]:
customers.tail()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS
99440,274fa6071e5e17fe303b9748641082c8,84732c5050c01db9b23e19ba39899398,6703,cotia,SP


In [17]:
customers.describe(include = "all")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN


In [23]:
#to check which columns has unique values
print("Unique customer_id:", customers["customer_id"].nunique())
print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())

Unique customer_id: 99441
Unique customer_unique_id: 96096


In [25]:
#to find duplicate rows in the table
print("Duplicate rows:", customers.duplicated().sum())

Duplicate rows: 0


In [27]:
#to find out how many customers have multiple customer_ids.
customer_id_counts = (
    customers.groupby("customer_unique_id")["customer_id"]
    .nunique()
)

print("Customers with multiple customer_ids:",
      (customer_id_counts > 1).sum())

print("\nMaximum customer_ids for one customer:",
      customer_id_counts.max())

print("\nDistribution:")
print(customer_id_counts.value_counts().sort_index())

Customers with multiple customer_ids: 2997

Maximum customer_ids for one customer: 17

Distribution:
customer_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64


In [29]:
#to analyze the geographic distribution
print("Unique ZIP-code prefixes:", customers["customer_zip_code_prefix"].nunique())
print("Unique cities:", customers["customer_city"].nunique())
print("Unique states:", customers["customer_state"].nunique())

Unique ZIP-code prefixes: 14994
Unique cities: 4119
Unique states: 27


In [31]:
customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [33]:
customers["customer_state"].value_counts(normalize=True).mul(100).round(2)

customer_state
SP    41.98
RJ    12.92
MG    11.70
RS     5.50
PR     5.07
SC     3.66
BA     3.40
DF     2.15
ES     2.04
GO     2.03
PE     1.66
CE     1.34
PA     0.98
MT     0.91
MA     0.75
MS     0.72
PB     0.54
PI     0.50
RN     0.49
AL     0.42
SE     0.35
TO     0.28
RO     0.25
AM     0.15
AC     0.08
AP     0.07
RR     0.05
Name: proportion, dtype: float64

In [35]:
customers["customer_city"].value_counts().head(20)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
niteroi                    849
santo andre                797
osasco                     746
santos                     713
goiania                    692
sao jose dos campos        691
fortaleza                  654
sorocaba                   633
recife                     613
florianopolis              570
Name: count, dtype: int64

In [37]:
city_state_counts = (
    customers.groupby("customer_city")["customer_state"]
    .nunique()
    .sort_values(ascending=False)
)

city_state_counts.head(20)

customer_city
sao francisco          4
sao domingos           4
santa maria            4
vera cruz              4
bonito                 4
planalto               4
boa esperanca          4
vicosa                 3
agua branca            3
mundo novo             3
cantagalo              3
sao joao do paraiso    3
sobradinho             3
praia grande           3
santa luzia            3
bom jardim             3
belem                  3
bom jesus              3
triunfo                3
itambe                 3
Name: customer_state, dtype: int64

In [39]:
zip_state_counts = (
    customers.groupby("customer_zip_code_prefix")["customer_state"]
    .nunique()
    .sort_values(ascending=False)
)

zip_state_counts.head(20)

customer_zip_code_prefix
1003     1
59071    1
59054    1
59056    1
59060    1
59062    1
59063    1
59064    1
59065    1
59066    1
59067    1
59068    1
59069    1
59070    1
59073    1
59104    1
59074    1
59075    1
59076    1
59078    1
Name: customer_state, dtype: int64

In [41]:
print("ZIP prefixes mapped to multiple states:",
      (zip_state_counts > 1).sum())

print("Maximum states for one ZIP prefix:",
      zip_state_counts.max())

ZIP prefixes mapped to multiple states: 0
Maximum states for one ZIP prefix: 1


In [43]:
print("Minimum ZIP prefix:", customers["customer_zip_code_prefix"].min())
print("Maximum ZIP prefix:", customers["customer_zip_code_prefix"].max())

Minimum ZIP prefix: 1003
Maximum ZIP prefix: 99990


In [45]:
customers["customer_zip_code_prefix"].describe()

count    99441.000000
mean     35137.474583
std      29797.938996
min       1003.000000
25%      11347.000000
50%      24416.000000
75%      58900.000000
max      99990.000000
Name: customer_zip_code_prefix, dtype: float64

In [47]:
customers["customer_zip_code_prefix"].astype(str).str.len().value_counts().sort_index()

customer_zip_code_prefix
4    23995
5    75446
Name: count, dtype: int64

In [53]:
#to Check no of unique city/state combinations 
city_state = customers[["customer_city", "customer_state"]]

print("Unique city-state combinations:", city_state.drop_duplicates().shape[0])
print("Duplicate city-state rows:", city_state.duplicated().sum())

Unique city-state combinations: 4310
Duplicate city-state rows: 95131


In [51]:
customer_state_counts = (
    customers.groupby("customer_unique_id")["customer_state"]
    .nunique()
)

print("Customers associated with multiple states:",
      (customer_state_counts > 1).sum())

print("Maximum states for one customer:",
      customer_state_counts.max())

Customers associated with multiple states: 39
Maximum states for one customer: 3


### **Profiling Findings — customers :** 

### **Structure**
* **99,441** rows and **5** columns.
* Columns consist of **4** categorical/object variables and **1** integer variable.
* **No missing values** were found.
* **No completely duplicated rows** were found.

### **Customer Identifiers**
* `customer_id` contains **99,441** unique values, making it unique across the table.
* `customer_unique_id` contains **96,096** unique customers.
* **2,997** customers have multiple `customer_id` records, with one customer having as many as **17**.
* **Key Takeaway:** `customer_unique_id` is the appropriate identifier when performing customer-level analysis.

### **Geographic Coverage**
* **14,994** unique ZIP-code prefixes.
* **4,119** unique cities across **27** state/federal-district codes.
* Customer distribution is highly concentrated geographically: **SP, RJ, and MG account for 66.60%** of all records (**São Paulo alone accounts for 41.98%**).

### **Geographic Data Quality**
* City names are not unique geographical identifiers on their own; there are **4,310** unique city–state combinations.
* All ZIP-code prefixes map to **exactly one state** in the dataset.
* ZIP prefixes range from **1,003 to 99,990**.
* **23,995** records have four-digit stored values, likely because leading zeros were dropped when stored as integers.
* **39** unique customers have records associated with multiple states (maximum of **3** states per customer).
* **Key Takeaway:** ZIP prefixes should be treated as **categorical/geographical codes**, not continuous numerical variables.

### **Data Profiling Order_items Table :**

In [65]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [67]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [69]:
order_items.tail()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72
112649,fffe41c64501cc87c801fd61db3f6244,1,350688d9dc1e75ff97be326363655e01,f7ccf836d21b2fb1de37564105216cc1,2018-06-12 17:10:13,43.00,12.79


In [75]:
order_items.describe(include = "all")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
count,112650,112650.000000,112650,112650,112650,112650.000000,112650.000000
unique,98666,NaN,32951,3095,93318,NaN,NaN
top,8272b63d03f5f79c56e9e4120aec44ef,NaN,aca2eb7d00ea1a7b8ebd4e68314663af,6560211a19b47992c3666cc44a7e94c0,2017-07-21 18:25:23,NaN,NaN
freq,21,NaN,527,2033,21,NaN,NaN
mean,NaN,1.197834,NaN,NaN,NaN,120.653739,19.990320
std,NaN,0.705124,NaN,NaN,NaN,183.633928,15.806405
min,NaN,1.000000,NaN,NaN,NaN,0.850000,0.000000
25%,NaN,1.000000,NaN,NaN,NaN,39.900000,13.080000
50%,NaN,1.000000,NaN,NaN,NaN,74.990000,16.260000
75%,NaN,1.000000,NaN,NaN,NaN,134.900000,21.150000


In [77]:
print("Duplicate rows:", order_items.duplicated().sum())

print("Duplicate (order_id, order_item_id):",
      order_items.duplicated(subset=['order_id', 'order_item_id']).sum())

print("Unique (order_id, order_item_id):",
      order_items[['order_id', 'order_item_id']].drop_duplicates().shape[0])

Duplicate rows: 0
Duplicate (order_id, order_item_id): 0
Unique (order_id, order_item_id): 112650


In [79]:
order_items.groupby('order_id')['order_item_id'].count().describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: order_item_id, dtype: float64

In [83]:
#to find how many order contains how many no of items
order_items.groupby('order_id')['order_item_id'].count().value_counts().sort_index()

order_item_id
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

In [85]:
print("Unique order_ids:", order_items['order_id'].nunique())
print("Unique product_ids:", order_items['product_id'].nunique())
print("Unique seller_ids:", order_items['seller_id'].nunique())

print("Order item_id min:", order_items['order_item_id'].min())
print("Order item_id max:", order_items['order_item_id'].max())

Unique order_ids: 98666
Unique product_ids: 32951
Unique seller_ids: 3095
Order item_id min: 1
Order item_id max: 21


In [87]:
order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date']
)

print("Min:", order_items['shipping_limit_date'].min())
print("Max:", order_items['shipping_limit_date'].max())
print("Unique dates:", order_items['shipping_limit_date'].nunique())

Min: 2016-09-19 00:15:34
Max: 2020-04-09 22:35:08
Unique dates: 93318


In [89]:
order_items.nlargest(10, 'shipping_limit_date')[
    ['order_id', 'order_item_id', 'product_id',
     'seller_id', 'shipping_limit_date', 'price', 'freight_value']
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
85729,c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44
85730,c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44
8643,13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,69.99,14.66
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,75.99,14.70
26104,3b61aab5de69abc1731138bd104a777f,1,6aa063e063f2ab982b471e58afe06d72,610f72e407cdd7caaa2f8167b0163fd8,2018-09-18 21:10:15,999.99,24.77
54967,7cfdf7265c9572fc7b7cbd3b9cc438b7,2,17e18b0c88a853dd6de3e48a7cfa9d9a,cee48807215b30a12ca2ca10ffb5f250,2018-09-14 12:30:56,20.00,19.25
11891,1afe384f199748cff7a42c9902065560,1,4c2a4020fcd651812100ebbeac1b2753,610f72e407cdd7caaa2f8167b0163fd8,2018-09-14 02:09:37,599.99,29.18
39543,59eaa904b3f0dbde2785ac1b27eccd18,1,61919b39651acb61ec24307ed8b9502d,f61c63d13f7cd800549d5acdd390ae72,2018-09-13 14:55:28,299.00,14.75
91384,cf5c8d9f52807cb2d2f0a0ff54c478da,6,a7bbff32c7321478b29f924301a1867d,dfc475d54e1b6dbeeb7d7d9bdaa63827,2018-09-12 13:24:27,16.90,8.99
93959,d4fae577806d683110e00e18a5e181be,4,7001d71d1ad858e07e5a341649412e76,f0b47fbbc6dee9aafe415a6e33051b3f,2018-09-12 03:15:36,49.99,3.57


In [91]:
order_items[order_items['shipping_limit_date'] >= '2019-01-01'].shape

(4, 7)

In [93]:
order_items[['price', 'freight_value']].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [95]:
print("Zero freight values:",
      (order_items['freight_value'] == 0).sum())

print("Zero freight percentage:",
      (order_items['freight_value'] == 0).mean() * 100)

Zero freight values: 383
Zero freight percentage: 0.33999112294718153


In [97]:
print("Items with price < 10:",
      (order_items['price'] < 10).sum())

print("Items with price > 1000:",
      (order_items['price'] > 1000).sum())

Items with price < 10: 1188
Items with price > 1000: 844


In [99]:
product_seller_counts = (
    order_items.groupby('product_id')['seller_id']
    .nunique()
)

print("Products sold by multiple sellers:",
      (product_seller_counts > 1).sum())

print("Maximum sellers for one product:",
      product_seller_counts.max())

Products sold by multiple sellers: 1225
Maximum sellers for one product: 8


In [101]:
seller_item_counts = order_items['seller_id'].value_counts()

print(seller_item_counts.describe())

print("\nTop 10 sellers:")
print(seller_item_counts.head(10))

count    3095.000000
mean       36.397415
std       119.193461
min         1.000000
25%         2.000000
50%         8.000000
75%        24.000000
max      2033.000000
Name: count, dtype: float64

Top 10 sellers:
seller_id
6560211a19b47992c3666cc44a7e94c0    2033
4a3ca9315b744ce9f8e9374361493884    1987
1f50f920176fa81dab994f9023523100    1931
cc419e0650a3c5ba77189a1882b7556a    1775
da8622b14eb17ae2831f4ac5b9dab84a    1551
955fee9216a65b617aa5c0531780ce60    1499
1025f0e2d44d7041d6cf58b6550e0bfa    1428
7c67e1448b00f6e969d365cea6b010ab    1364
ea8482cd71df3c1969d7b9473ff13abc    1203
7a67c85e85bb2ce8582c35f2203ad736    1171
Name: count, dtype: int64


In [103]:
product_item_counts = order_items['product_id'].value_counts()

print(product_item_counts.describe())

print("\nTop 10 products:")
print(product_item_counts.head(10))

count    32951.000000
mean         3.418713
std         10.619709
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        527.000000
Name: count, dtype: float64

Top 10 products:
product_id
aca2eb7d00ea1a7b8ebd4e68314663af    527
99a4788cb24856965c36a24e339b6058    488
422879e10f46682990de24d770e7f83d    484
389d119b48cf3043d311335e499d9c6b    392
368c6c730842d78016ad823897a372db    388
53759a2ecddad2bb87a079a1f1519f73    373
d1c427060a0f73f6b889a5c7c61f2ac4    343
53b36df67ebb7c41585e8d54d6772e08    323
154e7e31ebfa092203795c972e5804a6    281
3dd2a17168ec895c781a9191c1e95ad7    274
Name: count, dtype: int64


In [105]:
missing_orders = ~order_items['order_id'].isin(orders['order_id'])

print("Order items with missing order_id in orders:",
      missing_orders.sum())

print("Percentage:",
      missing_orders.mean() * 100)

Order items with missing order_id in orders: 0
Percentage: 0.0


In [107]:
missing_products = ~order_items['product_id'].isin(products['product_id'])

print("Order items with missing product_id in products:",
      missing_products.sum())

print("Percentage:",
      missing_products.mean() * 100)

Order items with missing product_id in products: 0
Percentage: 0.0


In [109]:
missing_sellers = ~order_items['seller_id'].isin(sellers['seller_id'])

print("Order items with missing seller_id in sellers:",
      missing_sellers.sum())

print("Percentage:",
      missing_sellers.mean() * 100)

Order items with missing seller_id in sellers: 0
Percentage: 0.0


In [111]:
product_seller_pairs = order_items[
    ['product_id', 'seller_id']
].drop_duplicates()

print("Unique product-seller pairs:",
      len(product_seller_pairs))

print("Average sellers per product:",
      product_seller_pairs.groupby('product_id')['seller_id'].nunique().mean())

Unique product-seller pairs: 34448
Average sellers per product: 1.0454310946557008


In [113]:
invalid_sequence = (
    order_items.groupby('order_id')['order_item_id']
    .agg(lambda x: set(x) != set(range(1, len(x) + 1)))
)

print("Orders with non-sequential item IDs:",
      invalid_sequence.sum())

Orders with non-sequential item IDs: 0


### **Profiling Findings — order_items :**


* **`order_items`** contains **112,650** rows and **7** columns, with **no missing values**.
* (**`order_id`**, **`order_item_id`**) forms a **composite primary key**; there are **no duplicate rows** or duplicate key combinations.

* There are **98,666** unique orders, **32,951** unique products, and **3,095** unique sellers.
* **`order_item_id`** ranges from **1–21** and is sequential within every order, with **no gaps**.

* Orders contain an average of **1.14 items**, and approximately **90.1%** of orders contain exactly **one item**; the maximum is **21 items**.
* **`order_items`** has **100% referential integrity** with `orders`, `products`, and `sellers`.
* **`shipping_limit_date`** was stored as object and converted to datetime; it ranges from **2016-09-19** to **2020-04-09**.

* Only **4 records (0.0036%)** have shipping-limit dates in **2019 or later**, making them extreme temporal outliers requiring investigation.
* **`price`** is strongly right-skewed, ranging from **0.85 to 6,735**, with mean **120.65** and median **74.99**.
* **`freight_value`** ranges from **0 to 409.68**, with mean **19.99** and median **16.26**; **383 records (0.34%)** have zero freight.
* **844 items (0.75%)** have `price` > 1,000, while **1,188 (1.05%)** have `price` < 10.

* **Seller activity** is highly right-skewed: median seller has **8** order items, while the most active seller has **2,033**.

* **Product activity** is also highly right-skewed: median product appears **once**, while the most frequent product appears **527 times**.
* **1,225 products (~3.72%)** are sold by multiple sellers, with a maximum of **8 sellers** per product.

* There are **34,448** unique product–seller combinations, with an average of **1.045 sellers per product**, indicating most products are associated with one seller.

### **Data Profiling products Table :**

In [127]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [129]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [131]:
products.tail()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0
32950,106392145fca363410d287a815be6de4,cama_mesa_banho,58.0,309.0,1.0,2083.0,12.0,2.0,7.0


In [133]:
products.describe(include="all")

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32951,32341,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
unique,32951,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,3029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,NaN,NaN,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,NaN,NaN,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,NaN,NaN,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,NaN,NaN,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,NaN,NaN,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000


In [135]:
products.nunique()

product_id                    32951
product_category_name            73
product_name_lenght              66
product_description_lenght     2960
product_photos_qty               19
product_weight_g               2204
product_length_cm                99
product_height_cm               102
product_width_cm                 95
dtype: int64

In [137]:
print(products[products['product_weight_g'] == 0].shape)

print("\nMissing-value combinations:")
print(
    products.isna().sum(axis=1)
    .value_counts()
    .sort_index()
)

(4, 9)

Missing-value combinations:
0    32340
4      610
8        1
Name: count, dtype: int64


In [139]:
print("Zero-weight products:")
print(
    products[products['product_weight_g'] == 0][
        ['product_id', 'product_category_name',
         'product_weight_g', 'product_length_cm',
         'product_height_cm', 'product_width_cm']
    ]
)

print("\nProduct with 8 missing values:")
print(products[products.isna().sum(axis=1) == 8])

Zero-weight products:
                             product_id product_category_name  \
9769   81781c0fed9fe1ad6e8c81fca1e1cb08       cama_mesa_banho   
13683  8038040ee2a71048d4bdbbdc985b69ab       cama_mesa_banho   
14997  36ba42dd187055e1fbe943b2d11430ca       cama_mesa_banho   
32079  e673e90efa65a5409ff4196c038bb5af       cama_mesa_banho   

       product_weight_g  product_length_cm  product_height_cm  \
9769                0.0               30.0               25.0   
13683               0.0               30.0               25.0   
14997               0.0               30.0               25.0   
32079               0.0               30.0               25.0   

       product_width_cm  
9769               30.0  
13683              30.0  
14997              30.0  
32079              30.0  

Product with 8 missing values:
                             product_id product_category_name  \
18851  5eb564652db742ff8f28759cd8d2652a                   NaN   

       product_name_lenght  produ

In [141]:
category_counts = products['product_category_name'].value_counts()

print(category_counts.describe())

print("\nTop 10 categories:")
print(category_counts.head(10))

count      73.000000
mean      443.027397
std       735.904904
min         1.000000
25%        31.000000
50%        94.000000
75%       400.000000
max      3029.000000
Name: count, dtype: float64

Top 10 categories:
product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64


In [143]:
missing_category = products['product_category_name'].isna()

print("Missing category:", missing_category.sum())

print("Missing category AND missing name length:",
      (missing_category & products['product_name_lenght'].isna()).sum())

print("Missing category AND missing description length:",
      (missing_category & products['product_description_lenght'].isna()).sum())

print("Missing category AND missing photos:",
      (missing_category & products['product_photos_qty'].isna()).sum())

Missing category: 610
Missing category AND missing name length: 610
Missing category AND missing description length: 610
Missing category AND missing photos: 610


In [145]:
physical_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

print(products[physical_cols].isna().sum())

print("\nRows with any physical attribute missing:")
print(products[products[physical_cols].isna().any(axis=1)][
    ['product_id'] + physical_cols
])

product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64

Rows with any physical attribute missing:
                             product_id  product_weight_g  product_length_cm  \
8578   09ff539a621711667c43eba6a3bd8466               NaN                NaN   
18851  5eb564652db742ff8f28759cd8d2652a               NaN                NaN   

       product_height_cm  product_width_cm  
8578                 NaN               NaN  
18851                NaN               NaN  


In [147]:
physical_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in physical_cols:
    print(f"\n{col}")
    print("<= 0:", (products[col] <= 0).sum())
    print("Negative:", (products[col] < 0).sum())


product_weight_g
<= 0: 4
Negative: 0

product_length_cm
<= 0: 0
Negative: 0

product_height_cm
<= 0: 0
Negative: 0

product_width_cm
<= 0: 0
Negative: 0


In [149]:
print(
    "Products never appearing in order_items:",
    (~products['product_id'].isin(order_items['product_id'])).sum()
)

print(
    "Order-item products missing from products:",
    (~order_items['product_id'].isin(products['product_id'])).sum()
)

Products never appearing in order_items: 0
Order-item products missing from products: 0


### **Profiling Findings — products :**

* **32,951** rows and **9** columns, with `product_id` uniquely identifying every product.
* **610 products (~1.85%)** have missing category, name length, description length, and photo count simultaneously.
* **2 products** have all four physical attributes missing; one of these is the completely incomplete product with **8 missing attributes**.
* **4 products** have **0 g** weight, all in `cama_mesa_banho` with identical **30×25×30 cm** dimensions — potential data-quality anomalies.
* No negative physical measurements exist, and all dimensions are positive when present.
* There are **73** product categories, with highly uneven category sizes.
* `cama_mesa_banho` is the largest category with **3,029** products.
* Product description length is right-skewed, ranging from **4 to 3,992** characters.
* Products have **1–20** photos, with median **1** and mean **2.19**.
* Product weight is strongly right-skewed, ranging from **0 to 40,425 g**.
* Product dimensions range from **7–105 cm** length, **2–105 cm** height, and **6–118 cm** width.
* Every product appears in `order_items` and every product referenced by `order_items` exists in `products`, giving complete bidirectional coverage.

### **Data Profiling sellers Table :**

In [158]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [160]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [162]:
sellers.tail()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS
3094,9e25199f6ef7e7c347120ff175652c3b,12051,taubate,SP


In [166]:
sellers.describe(include="all")

,seller_id,seller_zip_code_prefix,seller_city,seller_state
count,3095,3095.000000,3095,3095
unique,3095,NaN,611,23
top,3442f8959a84dea7ee197c632cb2df15,NaN,sao paulo,SP
freq,1,NaN,694,1849
mean,NaN,32291.059451,NaN,NaN
std,NaN,32713.453830,NaN,NaN
min,NaN,1001.000000,NaN,NaN
25%,NaN,7093.500000,NaN,NaN
50%,NaN,14940.000000,NaN,NaN
75%,NaN,64552.500000,NaN,NaN


In [168]:
print("Duplicate rows:", sellers.duplicated().sum())
print("Duplicate seller IDs:", sellers['seller_id'].duplicated().sum())

Duplicate rows: 0
Duplicate seller IDs: 0


In [170]:
print("Unique city-state combinations:",
      sellers[['seller_city', 'seller_state']].drop_duplicates().shape[0])

Unique city-state combinations: 636


In [172]:
city_state_counts = sellers.groupby('seller_city')['seller_state'].nunique()

print("Cities associated with multiple states:",
      (city_state_counts > 1).sum())

print("Maximum states for one city:",
      city_state_counts.max())

Cities associated with multiple states: 23
Maximum states for one city: 3


In [174]:
zip_state_counts = sellers.groupby('seller_zip_code_prefix')['seller_state'].nunique()

print("ZIP prefixes associated with multiple states:",
      (zip_state_counts > 1).sum())

print("Maximum states for one ZIP prefix:",
      zip_state_counts.max())

ZIP prefixes associated with multiple states: 17
Maximum states for one ZIP prefix: 3


In [176]:
print("Sellers appearing in order_items:",
      order_items['seller_id'].nunique())

print("Sellers never appearing in order_items:",
      sellers['seller_id'].nunique() -
      order_items['seller_id'].nunique())

print("Order-item seller IDs missing from sellers:",
      (~order_items['seller_id'].isin(sellers['seller_id'])).sum())

Sellers appearing in order_items: 3095
Sellers never appearing in order_items: 0
Order-item seller IDs missing from sellers: 0


In [178]:
zip_city_counts = sellers.groupby('seller_zip_code_prefix')['seller_city'].nunique()

print("ZIP prefixes associated with multiple cities:",
      (zip_city_counts > 1).sum())

print("Maximum cities for one ZIP prefix:",
      zip_city_counts.max())

ZIP prefixes associated with multiple cities: 34
Maximum cities for one ZIP prefix: 2


### **Profiling Findings — sellers :**

* **Shape:** **3,095** rows × **4** columns.
* **Missing values:** No missing values in any column.
* **`seller_id`:** **3,095** unique values; no duplicate IDs; strong primary-key candidate.
* **Duplicate rows:** **0**.
* **Seller cities:** **611** unique cities.
* **Seller states:** **23** unique states.
* **City–state combinations:** **636**.
* **City–state consistency:** **23** city names occur across multiple states, with a maximum of **3** states for one city name. This is not necessarily an error because city names can repeat across states.
* **ZIP prefixes:** Range from **1,001 to 99,730**.
* **ZIP prefix–state consistency:** **17** ZIP prefixes occur across multiple states, with a maximum of **3** states.
* **ZIP prefix–city consistency:** **34** ZIP prefixes occur across multiple cities, with a maximum of **2** cities.
* **Geographical interpretation:** `seller_zip_code_prefix` is a coarse geographical identifier and should not be treated as an exact location or continuous numerical variable.
* **Seller concentration:** São Paulo (SP) contains **1,849** sellers, approximately **59.7%** of all sellers.
* **City concentration:** São Paulo city contains **694** sellers, approximately **22.4%** of all sellers.
* **Seller ↔ `order_items` integrity:** All **3,095** sellers appear in `order_items`.
* **Sellers with no order items:** **0**.
* **Order-item seller IDs missing from sellers:** **0**.
* **Overall relationship integrity:** Complete consistency between `sellers` and `order_items`.

### **Data Profiling order_payments Table :**

In [187]:
order_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [189]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [191]:
order_payments.tail()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54
103885,28bbae6599b09d39ca406b747b6632b1,1,boleto,1,191.58


In [193]:
order_payments.describe(include="all")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103886,103886.000000,103886,103886.000000,103886.000000
unique,99440,NaN,5,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092679,NaN,2.853349,154.100380
std,NaN,0.706584,NaN,2.687051,217.494064
min,NaN,1.000000,NaN,0.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.790000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.837500


In [197]:
print("Unique order IDs:", order_payments['order_id'].nunique())
print("Duplicate rows:", order_payments.duplicated().sum())

Unique order IDs: 99440
Duplicate rows: 0


In [199]:
print(
    "Duplicate order_id + payment_sequential:",
    order_payments.duplicated(
        subset=['order_id', 'payment_sequential']
    ).sum()
)

Duplicate order_id + payment_sequential: 0


In [201]:
order_payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [203]:
order_payments['payment_type'].nunique()

5

In [205]:
print(order_payments['payment_sequential'].describe())
print("\nUnique values:")
print(sorted(order_payments['payment_sequential'].unique()))

count    103886.000000
mean          1.092679
std           0.706584
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          29.000000
Name: payment_sequential, dtype: float64

Unique values:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


In [207]:
print(order_payments['payment_installments'].describe())
print("\nUnique values:")
print(sorted(order_payments['payment_installments'].unique()))

count    103886.000000
mean          2.853349
std           2.687051
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

Unique values:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24]


In [209]:
print("Zero installments:",
      (order_payments['payment_installments'] == 0).sum())

print("Zero-installment percentage:",
      (order_payments['payment_installments'] == 0).mean() * 100)

Zero installments: 2
Zero-installment percentage: 0.0019251872244575785


In [211]:
order_payments.loc[
    order_payments['payment_installments'] == 0,
    'payment_type'
].value_counts()

payment_type
credit_card    2
Name: count, dtype: int64

In [213]:
print(order_payments['payment_value'].describe())

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64


In [215]:
print("Zero payment values:",
      (order_payments['payment_value'] == 0).sum())

print("Negative payment values:",
      (order_payments['payment_value'] < 0).sum())

Zero payment values: 9
Negative payment values: 0


In [217]:
missing_payment_order = orders.loc[
    ~orders['order_id'].isin(order_payments['order_id']),
    'order_id'
]

missing_payment_order

30710    bfbd0f9bdef84302105ad712db648a6c
Name: order_id, dtype: object

In [219]:
orders.loc[
    orders['order_id'].isin(missing_payment_order),
    ['order_id', 'order_status']
]

,order_id,order_status
30710,bfbd0f9bdef84302105ad712db648a6c,delivered


### **Profiling Findings — order_payments :**


* **Shape:** **103,886 × 5**.
* **Missing values:** None.
* **Unique orders:** **99,440**.
* **Duplicate rows:** **0**.
* **Duplicate `order_id` + `payment_sequential`:** **0**.
* **Composite key:** `order_id` + `payment_sequential`.
* **Payment types:** **5**.
* **Credit card:** **76,795 (~73.9%)**.
* **Boleto:** **19,784 (~19.0%)**.
* **Voucher:** **5,775 (~5.6%)**.
* **Debit card:** **1,529 (~1.5%)**.
* **`not_defined`:** **3** records.
* **`payment_sequential`:** **1–29**; median = **1**.
* **`payment_installments`:** **0–24**; median = **1**.
* **Zero installments:** **2** records (**0.0019%**), both credit-card payments.
* **`payment_value`:** **0–13,664.08**.
* **Zero payment values:** **9 (0.0087%)**.
* **Negative payment values:** **0**.
* **Payment value distribution:** Right-skewed (mean **154.10** vs median **100.00**).
* **Maximum payment:** **13,664.08**.
* **Order-payment completeness:** **99,440 of 99,441** orders have payment records.
* **Missing payment relationship:** **1** delivered order has no payment record.
* **Overall assessment:** The table is structurally clean with a few extremely rare anomalies and one referential-integrity issue.

### **Data Profiling order_reviews Table :**

In [227]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [229]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [231]:
order_reviews.tail()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13
99223,efe49f1d6f951dd88b51e6ccd4cc548f,90531360ecb1eec2a1fbb265a0db0508,1,NaN,"meu produto chegou e ja tenho que devolver, po...",2017-07-03 00:00:00,2017-07-03 21:01:49


In [235]:
order_reviews.describe(include="all")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
count,99224,99224,99224.000000,11568,40977,99224,99224
unique,98410,98673,NaN,4527,36159,636,98248
top,7b606b0d57b078384f0b58eac1d41d78,c88b1d1b157a9999ce368f218a407141,NaN,Recomendo,Muito bom,2017-12-19 00:00:00,2017-06-15 23:21:05
freq,3,3,NaN,423,230,463,4
mean,NaN,NaN,4.086421,NaN,NaN,NaN,NaN
std,NaN,NaN,1.347579,NaN,NaN,NaN,NaN
min,NaN,NaN,1.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,4.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN


In [237]:
print("Unique review IDs:", order_reviews['review_id'].nunique())
print("Unique order IDs:", order_reviews['order_id'].nunique())
print("Duplicate rows:", order_reviews.duplicated().sum())
print("Duplicate review IDs:", order_reviews['review_id'].duplicated().sum())

Unique review IDs: 98410
Unique order IDs: 98673
Duplicate rows: 0
Duplicate review IDs: 814


In [239]:
review_id_counts = order_reviews['review_id'].value_counts()

print("Review IDs appearing more than once:",
      (review_id_counts > 1).sum())

print("\nMaximum occurrences of one review_id:",
      review_id_counts.max())

Review IDs appearing more than once: 789

Maximum occurrences of one review_id: 3


In [241]:
order_review_counts = order_reviews['order_id'].value_counts()

print("Orders with multiple review records:",
      (order_review_counts > 1).sum())

print("Maximum reviews for one order:",
      order_review_counts.max())

Orders with multiple review records: 547
Maximum reviews for one order: 3


In [243]:
duplicate_reviews = order_reviews[
    order_reviews['review_id'].duplicated(keep=False)
].sort_values('review_id')

duplicate_reviews.head(20)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [245]:
order_reviews[
    order_reviews['review_id'].duplicated(keep=False)
].sort_values('review_id')[
    ['review_id', 'order_id', 'review_score',
     'review_comment_title', 'review_comment_message',
     'review_creation_date', 'review_answer_timestamp']
].head(10)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [247]:
order_reviews['review_score'].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [249]:
order_reviews['review_score'].describe()

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

In [251]:
print("Review creation date range:")
print(order_reviews['review_creation_date'].min(),
      "to",
      order_reviews['review_creation_date'].max())

print("\nReview answer timestamp range:")
print(order_reviews['review_answer_timestamp'].min(),
      "to",
      order_reviews['review_answer_timestamp'].max())

Review creation date range:
2016-10-02 00:00:00 to 2018-08-31 00:00:00

Review answer timestamp range:
2016-10-07 18:32:28 to 2018-10-29 12:27:35


In [253]:
creation = pd.to_datetime(order_reviews['review_creation_date'])
answer = pd.to_datetime(order_reviews['review_answer_timestamp'])

print("Answers before review creation:",
      (answer < creation).sum())

Answers before review creation: 0


In [255]:
response_time = answer - creation

print(response_time.describe())

count                        99224
mean     3 days 03:34:33.029700475
std      9 days 21:21:40.258026234
min                0 days 02:08:29
25%         1 days 00:07:00.750000
50%         1 days 16:11:55.500000
75%                3 days 02:29:08
max              518 days 16:46:52
dtype: object


In [257]:
print("Review orders:", order_reviews['order_id'].nunique())
print("Order IDs in reviews missing from orders:",
      (~order_reviews['order_id'].isin(orders['order_id'])).sum())

print("Orders with no review:",
      (~orders['order_id'].isin(order_reviews['order_id'])).sum())

Review orders: 98673
Order IDs in reviews missing from orders: 0
Orders with no review: 768


### **Profiling Findings — order_reviews :**

* **Shape:** **99,224** rows × **7** columns.
* **Missing values:** Only `review_comment_title` and `review_comment_message` contain missing values.
* **Review title missing:** **87,656 (~88.3%)**.
* **Review message missing:** **58,247 (~58.7%)**.
* **`review_score`:** No missing values; valid range of **1–5**.
* **Mean review score:** **4.086**; median = **5**.
* **Review distribution:** 4- and 5-star reviews account for **~77.1%**, indicating strong positive-rating concentration.
* **Duplicate rows:** **0**.
* **`review_id`:** **98,410** unique values; **789** IDs occur multiple times, with a maximum frequency of **3**. Therefore, `review_id` is not a unique key.
* **`order_id`:** **98,673** unique orders; **547** orders have multiple review records, with a maximum of **3**.
* **Review creation dates:** **2016-10-02 to 2018-08-31**.
* **Review answer timestamps:** **2016-10-07 to 2018-10-29**.
* **Temporal consistency:** **0** answers occur before review creation.
* **Response time:** Median **~1.67 days** and mean **~3.15 days**; distribution is strongly right-skewed.
* **Maximum response time:** **~518.7 days**, representing an extreme long-tail observation.
* **Order relationship:** All reviewed orders exist in `orders`; **0** orphan review records.
* **Orders without reviews:** **768 (~0.8%)**.
* **Date columns:** Currently stored as object; should be converted to datetime during preprocessing.

### **Data Profiling geolocation Table :**

In [266]:
geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


In [268]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [270]:
geolocation.tail()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS
1000162,99950,-28.070104,-52.018658,tapejara,RS


In [272]:
geolocation.describe(include ="all")

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
count,1.000163e+06,1.000163e+06,1.000163e+06,1000163,1000163
unique,NaN,NaN,NaN,8011,27
top,NaN,NaN,NaN,sao paulo,SP
freq,NaN,NaN,NaN,135800,404268
mean,3.657417e+04,-2.117615e+01,-4.639054e+01,NaN,NaN
std,3.054934e+04,5.715866e+00,4.269748e+00,NaN,NaN
min,1.001000e+03,-3.660537e+01,-1.014668e+02,NaN,NaN
25%,1.107500e+04,-2.360355e+01,-4.857317e+01,NaN,NaN
50%,2.653000e+04,-2.291938e+01,-4.663788e+01,NaN,NaN
75%,6.350400e+04,-1.997962e+01,-4.376771e+01,NaN,NaN


In [274]:
geolocation.duplicated().sum()

261831

In [276]:
print("Unique ZIP prefixes:", geolocation['geolocation_zip_code_prefix'].nunique())
print("Unique ZIP-city combinations:",
      geolocation[['geolocation_zip_code_prefix', 'geolocation_city']].drop_duplicates().shape[0])
print("Unique ZIP-state combinations:",
      geolocation[['geolocation_zip_code_prefix', 'geolocation_state']].drop_duplicates().shape[0])

Unique ZIP prefixes: 19015
Unique ZIP-city combinations: 27907
Unique ZIP-state combinations: 19023


In [278]:
zip_multiple_states = (
    geolocation.groupby('geolocation_zip_code_prefix')['geolocation_state']
    .nunique()
)

print("ZIP prefixes associated with multiple states:",
      (zip_multiple_states > 1).sum())

print(zip_multiple_states[zip_multiple_states > 1].sort_values(ascending=False).head(20))

ZIP prefixes associated with multiple states: 8
geolocation_zip_code_prefix
2116     2
4011     2
21550    2
23056    2
72915    2
78557    2
79750    2
80630    2
Name: geolocation_state, dtype: int64


In [280]:
print("Latitude < -35:", (geolocation['geolocation_lat'] < -35).sum())
print("Latitude > 5:", (geolocation['geolocation_lat'] > 5).sum())

print("Longitude < -75:", (geolocation['geolocation_lng'] < -75).sum())
print("Longitude > -30:", (geolocation['geolocation_lng'] > -30).sum())

Latitude < -35: 3
Latitude > 5: 26
Longitude < -75: 4
Longitude > -30: 22


In [282]:
lat_outliers = geolocation[
    (geolocation['geolocation_lat'] < -35) |
    (geolocation['geolocation_lat'] > 5)
]

lng_outliers = geolocation[
    (geolocation['geolocation_lng'] < -75) |
    (geolocation['geolocation_lng'] > -30)
]

print("Latitude outliers:")
display(lat_outliers)

print("\nLongitude outliers:")
display(lng_outliers)

Latitude outliers:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG
695377,45936,38.323939,-6.775035,itabatan,BA



Longitude outliers:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG
695377,45936,38.323939,-6.775035,itabatan,BA


In [284]:
city_multiple_states = (
    geolocation.groupby('geolocation_city')['geolocation_state']
    .nunique()
)

print("Cities associated with multiple states:",
      (city_multiple_states > 1).sum())

print(
    city_multiple_states[city_multiple_states > 1]
    .sort_values(ascending=False)
    .head(20)
)

Cities associated with multiple states: 362
geolocation_city
bom jesus          6
sao domingos       6
são domingos       5
sao pedro          5
boa esperança      5
boa esperanca      5
santa luzia        5
santa inês         4
campo grande       4
planalto           4
bonito             4
santa helena       4
santa ines         4
sao francisco      4
santa maria        4
vera cruz          4
alto alegre        4
santa terezinha    4
arapua             3
cantagalo          3
Name: geolocation_state, dtype: int64


In [286]:
print(
    "Unique city-state combinations:",
    geolocation[['geolocation_city', 'geolocation_state']]
    .drop_duplicates()
    .shape[0]
)

Unique city-state combinations: 8463


In [288]:
customer_zips = set(customers['customer_zip_code_prefix'].unique())
seller_zips = set(sellers['seller_zip_code_prefix'].unique())
geo_zips = set(geolocation['geolocation_zip_code_prefix'].unique())

print("Customer ZIP prefixes:", len(customer_zips))
print("Seller ZIP prefixes:", len(seller_zips))
print("Geolocation ZIP prefixes:", len(geo_zips))

print("Customer ZIPs missing in geolocation:",
      len(customer_zips - geo_zips))

print("Seller ZIPs missing in geolocation:",
      len(seller_zips - geo_zips))

Customer ZIP prefixes: 14994
Seller ZIP prefixes: 2246
Geolocation ZIP prefixes: 19015
Customer ZIPs missing in geolocation: 157
Seller ZIPs missing in geolocation: 7


In [290]:
zip_city_state = (
    geolocation
    .groupby([
        'geolocation_zip_code_prefix',
        'geolocation_city',
        'geolocation_state'
    ])
    .agg(
        unique_lat=('geolocation_lat', 'nunique'),
        unique_lng=('geolocation_lng', 'nunique'),
        rows=('geolocation_zip_code_prefix', 'size')
    )
)

print("ZIP-city-state groups:", len(zip_city_state))

print(
    "Groups with multiple latitude values:",
    (zip_city_state['unique_lat'] > 1).sum()
)

print(
    "Groups with multiple longitude values:",
    (zip_city_state['unique_lng'] > 1).sum()
)

ZIP-city-state groups: 27912
Groups with multiple latitude values: 25086
Groups with multiple longitude values: 25085


### **Profiling Findings — geolocation :**

* The table contains **1,000,163** rows and **5** columns, with **no missing values**.

* It contains **19,015** unique ZIP-code prefixes, **8,011** unique city names, **27** states, and **8,463** unique city-state combinations.

* There are **261,831** exact duplicate rows, representing approximately **26.2%** of the dataset.

* The table is **not uniquely identifiable by ZIP prefix**; multiple records exist for the same ZIP prefix.

* There are **27,907** unique ZIP-city combinations and **19,023** unique ZIP-state combinations.
* **8** ZIP prefixes are associated with multiple states, indicating minor geographic inconsistencies.
* **362** city names occur across multiple states. This is largely expected because Brazilian cities can share names across states; therefore, **city + state** is a more reliable geographic identifier than **city alone**.

* There are **27,912** ZIP-city-state groups, and approximately **25,086** have multiple latitude values and **25,085** have multiple longitude values, confirming that multiple geographic observations commonly exist within the same ZIP-city-state area.

* Latitude and longitude contain a very small number of extreme geographic anomalies: **29 latitude** and **26 longitude** outlier records. Several correspond to coordinates clearly inconsistent with their associated Brazilian city/state.

* The geolocation table provides high coverage of marketplace participants' ZIP prefixes: **98.95%** of unique customer ZIP prefixes are present.
* **99.69%** of unique seller ZIP prefixes are present.

* Therefore, the table is suitable for geographic enrichment of customer and seller data, but the raw table should **not be directly joined without aggregation** because multiple geolocation records per ZIP can create row multiplication.

* City names also contain text normalization variations, such as accented and unaccented versions (**são/sao**, **inês/ines**), which may require normalization during preprocessing.

### **Data Profiling category_name_translation Table :**

In [302]:
category_translation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [304]:
category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [306]:
category_translation.tail()

,product_category_name,product_category_name_english
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
70,seguros_e_servicos,security_and_services


In [308]:
category_translation.describe(include = "all")

,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


In [312]:
print("Duplicate rows:", category_translation.duplicated().sum())

print("Unique Portuguese categories:",
      category_translation['product_category_name'].nunique())

print("Unique English categories:",
      category_translation['product_category_name_english'].nunique())

Duplicate rows: 0
Unique Portuguese categories: 71
Unique English categories: 71


In [314]:
print(
    "Portuguese categories with multiple English translations:",
    (category_translation
     .groupby('product_category_name')['product_category_name_english']
     .nunique() > 1).sum()
)

Portuguese categories with multiple English translations: 0


In [316]:
product_categories = set(products['product_category_name'].dropna().unique())
translation_categories = set(category_translation['product_category_name'])

print("Unique product categories:", len(product_categories))
print("Translated categories:", len(translation_categories))

print(
    "Product categories missing from translation:",
    len(product_categories - translation_categories)
)

print(
    "Translation categories not in products:",
    len(translation_categories - product_categories)
)

Unique product categories: 73
Translated categories: 71
Product categories missing from translation: 2
Translation categories not in products: 0


In [318]:
missing_categories = (
    product_categories - translation_categories
)

print("Categories missing from translation:")
print(missing_categories)

Categories missing from translation:
{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


### **Profiling Findings — category_name_translation :**

* The table contains **71** rows and **2** columns with **no missing values**.

* Both columns are categorical/text fields: `product_category_name` and `product_category_name_english`.

* There are **no duplicate rows**.

* All **71** Portuguese category names are unique.

* All **71** English category names are unique.

* The table provides a **one-to-one mapping** between Portuguese and English product categories; no Portuguese category has multiple English translations.

* The `products` table contains **73** unique product categories, of which **71** are present in the translation table, giving **97.26%** unique-category coverage.
* **2** product categories are missing from the translation table:
  * `pc_gamer`
  * `portateis_cozinha_e_preparadores_de_alimentos`

* All **71** categories in the translation table are actually present in the `products` table; there are **no unused translation entries**.

* Therefore, the table is a **clean reference/lookup table**, but translation coverage is incomplete for **2** product categories, which will need to be handled during preprocessing if English category names are required.